# 音声生成ノートブック（Kaggle GPU / CPU）

**目的**：台本の全セリフを、セリフ単位で合成する。止まっても続きから再開でき、変更したセリフだけを作り直す。

- 入力：`script/dialogue.json`（台本）, `config/voice_config.yaml`（声の設定）／出力：`audio/raw/*.wav`, `audio/raw/manifest.json`, `logs/generate_audio.jsonl`
- GPU があれば `--device cuda` で合成する。**GPU が使えない場合**は自動で CPU を使う
- **セッション切断への対策**：manifest を1行ごとに保存する。再接続したら同じセルをもう一度実行すれば、生成済みの行は飛ばされる。
  セッションが終わる前に、最後のセルで `/kaggle/working/gene_quest_audio.zip` を作り、Output からダウンロードする
- **エラー時**：特定の行だけ失敗した場合は `--line SC0302-03 --force` でその行だけ作り直す
- API キーは不要。認証が必要なサービスを使う場合も、ノートブックに直接書かず Kaggle の Secrets を使う

In [ ]:
import os, sys, subprocess, shutil, platform
from pathlib import Path
ON_KAGGLE = Path("/kaggle").exists()
# プロジェクトの場所：Kaggle では Dataset として追加した production/gene-quest を /kaggle/working にコピーして使う
PROJECT = Path(os.environ.get("GQ_PROJECT", "/kaggle/working/gene-quest" if ON_KAGGLE else Path.cwd().parent))
SRC = Path("/kaggle/input/gene-quest") if ON_KAGGLE else None
if ON_KAGGLE and not PROJECT.exists() and SRC and SRC.exists():
    shutil.copytree(SRC, PROJECT)
print("project:", PROJECT, "exists:", PROJECT.exists())

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "style-bert-vits2", "kokoro-onnx", "misaki[ja]", "fugashi", "unidic-lite",
                "pyloudnorm", "soundfile", "pyyaml", "jaconv"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "unidic"])
subprocess.run(["bash", str(PROJECT / "scripts" / "download_models.sh")], check=True)

In [ ]:
try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print("device:", device)
r = subprocess.run([sys.executable, "scripts/generate_audio.py", "--device", device], cwd=PROJECT)
print("exit", r.returncode)

In [ ]:
# 仕上げ（音量・EQ）とタイムラインは CPU で十分
subprocess.run([sys.executable, "scripts/process_audio.py"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "scripts/build_timeline.py"], cwd=PROJECT, check=True)

In [ ]:
if ON_KAGGLE:
    shutil.make_archive("/kaggle/working/gene_quest_audio", "zip", PROJECT, "audio")
    print("→ /kaggle/working/gene_quest_audio.zip を Output からダウンロード")